# Release validation on CPU
Portable inference regression using existing weights. Set SPV_REPO and optional SPV_MANIFEST, SPV_IMAGES, SPV_LABELS and SPV_VALIDATION_OUTPUT. This notebook was edited but not executed in the present environment. Historical executed notebook 06 remains unchanged.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from datetime import datetime, timezone
REPO = Path(os.environ.get("SPV_REPO", Path.cwd())).expanduser().resolve()
if not (REPO / "src/spv").is_dir() or not (REPO / "configs/config.yaml").is_file():
    raise RuntimeError("Set SPV_REPO to the project folder containing src/spv and configs/config.yaml")
sys.path.insert(0, str(REPO / "src"))
from spv.common import load_config
cfg = load_config(REPO / "configs/config.yaml")
MANIFEST = Path(os.environ.get("SPV_MANIFEST", REPO / "runs/manifest.json"))
IMAGES = Path(os.environ.get("SPV_IMAGES", cfg["paths"]["team"] / "images"))
LABELS = Path(os.environ.get("SPV_LABELS", cfg["paths"]["team"] / "labels.csv"))
OUTPUT = Path(os.environ.get("SPV_VALIDATION_OUTPUT", REPO / "review_runs" / datetime.now(timezone.utc).strftime("cpu_%Y%m%dT%H%M%S%fZ")))
if OUTPUT.exists():
    raise RuntimeError("Choose a new SPV_VALIDATION_OUTPUT; prior outputs are preserved")
print("Project:", REPO, "Images:", IMAGES, "New output:", OUTPUT)


In [ ]:
command = [sys.executable, str(REPO / "tools/validate_release.py"), "--manifest", str(MANIFEST), "--images", str(IMAGES), "--labels", str(LABELS), "--output", str(OUTPUT), "--threads", "4"]
# No training. CPU inference; no historical result directory is overwritten.
subprocess.run(command, cwd=str(REPO), check=True)


In [ ]:
summary = json.loads((OUTPUT / "summary.json").read_text())
print(json.dumps(summary, indent=2))
print("This is regression on previously inspected photos, not a fresh independent test.")
